# Student Health Prediction Competition
## Data loading

In [ ]:
import numpy as np
import pandas as pd
import os

RUNS_ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
__DATA_DIR__ = '/kaggle/input/competitions/playground-series-s6e7/'

def get_data_path(file_name=None):
    if RUNS_ON_KAGGLE:
        return __DATA_DIR__ + (f'/{file_name}' if file_name else '')
    return os.path.join(os.getcwd(), 'data') + (f'/{file_name}' if file_name else '')

def get_output_path(file_name=None):
    if RUNS_ON_KAGGLE:
        return '/kaggle/working' + (f'/{file_name}' if file_name else '')
    return f'./data/{file_name}' if file_name else ''

def get_embedding_path(file_name=None):
    if RUNS_ON_KAGGLE:
        return '/kaggle/working' + (f'/{file_name}' if file_name else '')
    return os.path.join(os.getcwd(), 'embeddings') + (f'/{file_name}' if file_name else '')



In [ ]:
train_df = pd.read_csv(get_data_path('train.csv'), sep=',')
train_df.head()

## Check for Missing or Incomplete Data

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.isnull().sum()

In [ ]:
nan_percent = {}

for col in train_df.columns:
    nan_percent[col] = round(train_df[col].isna().mean() * 100, 2)

nan_percent_df = (
    pd.DataFrame.from_dict(
        nan_percent,
        orient="index",
        columns=["NAN_Percent"]
    )
    .sort_values("NAN_Percent", ascending=False)
)

print(nan_percent_df)

We have a signifficant number of missing values.

## Data Types Check

In [ ]:
print(train_df.dtypes)

Categorical features are being treated as `dtype = object`. Let's cast them to categories:

In [ ]:
cat_features = ['health_condition', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

for feature in cat_features:
    train_df[feature] = train_df[feature].astype('category')

train_df.dtypes

## Encoding of Categorical Features

The following attributes contain categorical data:

* `health_condition`
* `diet_type`
* `stress_level`
* `sleep_quality`
* `physical_activity_level`
* `smoking_alcohol`
* `gender`

We need to encode them in order to make sure models treat these data correctly. Let's start by exploring the possible values they take:

In [ ]:
for feature in cat_features:
    print(f"{feature}: {train_df[feature].unique()}")

In [ ]:
for feature in cat_features:
    print(f"{feature}: {train_df[feature].cat.categories.values}")

We notice some data contains NaN values. Let's see how many records are missing:

### Imputation of missing categorical values

In [ ]:
for cat in cat_features:
    if cat == 'health_condition':
        continue
    train_df[cat] = (
        train_df[cat]
            .cat.add_categories(["Missing"])
            .fillna("Missing")
    )

In [ ]:
train_df[cat_features].isna().sum()

We have successfully imputed `NaN` values in categorical features

## Imputation of Missing Numerical Values

In [ ]:
from sklearn.impute import SimpleImputer

# sleep_duration is our strongest predictor (highest Mutual Information), so a naive
# global-median imputation would systematically push ~11% of rows toward the
# "at-risk" class (the median sits right in that range). We instead:
#  1. Flag which rows had a missing value, so the model can distinguish real vs imputed data.
#  2. Impute using the median *within each sleep_quality group* (sleep_quality is already
#     NaN-free at this point and its groups differ meaningfully in sleep_duration:
#     ~6.45h for "poor" vs ~7.54h for "good"), falling back to the global median
#     for any group that has no data.
train_df['sleep_duration_was_missing'] = train_df['sleep_duration'].isna().astype(int)

train_df['sleep_duration'] = (
    train_df.groupby('sleep_quality', observed=True)['sleep_duration']
        .transform(lambda s: s.fillna(s.median()))
)
train_df['sleep_duration'] = train_df['sleep_duration'].fillna(train_df['sleep_duration'].median())

num_features = train_df.columns.difference(cat_features + ['id'])

remaining_num_features = num_features.difference(['sleep_duration', 'sleep_duration_was_missing'])

imputer = SimpleImputer(strategy="median")

train_df[remaining_num_features] = imputer.fit_transform(train_df[remaining_num_features])

In [ ]:
train_df.isna().sum()

Now our dataset is clean from missing values.

## Exploratory Data Analysis
### Basic Statistics

In [ ]:
train_df.describe()

### Box Plots

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

def make_box_plots(df, cols=4, rows=-1):
    if (rows == -1):
        rows = math.ceil(len(df.columns) / cols)
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=df.columns
    )

    i = 1
    j = 1

    for col in df.columns:
        fig.add_trace(go.Box(y=df[col], name=col), row=i, col=j)
        if j < cols:
            j += 1
        else:
            j = 1
            i += 1
    fig.update_layout(
        autosize=True,
        width=1200,
        height=1200,
        margin=dict(
            l=50,
            r=50,
            b=100,
            t=100,
            pad=4
        ),
    )

    for annotation in fig['layout']['annotations']:
        annotation['y'] += 0.005
    fig.show()


def make_histograms(df, cols=4, rows=-1):
    if (rows == -1):
        rows = math.ceil(len(df.columns) / cols)
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=df.columns
    )

    i = 1
    j = 1

    for col in df.columns:
        fig.add_trace(go.Histogram(x=df[col], name=col), row=i, col=j)
        if j < cols:
            j += 1
        else:
            j = 1
            i += 1
    fig.update_layout(
        autosize=True,
        width=1200,
        height=1200,
        margin=dict(
            l=50,
            r=50,
            b=100,
            t=100,
            pad=4
        )
    )

    for annotation in fig['layout']['annotations']:
        annotation['y'] += 0.005

    fig.show()

In [ ]:
make_box_plots(train_df[ train_df.columns.difference(cat_features)])

### Histograms

In [ ]:
make_histograms(train_df)

Histograms show we have unbalanced classes  in the target `health_condition` variable. This will be important later on.

We can also see how numerical features are mostly normally distributed, but it looks like some outliers are present. Let's analyze them.

### Removal of outliers from numerical features

In [ ]:
from scipy import stats

outlier_features = num_features.difference(['id', 'sleep_duration_was_missing'])

z_scores = np.abs(stats.zscore(train_df[outlier_features]))
outlier_mask = (z_scores < 3).all(axis=1)

print(f"Removing {(~outlier_mask).sum()} outlier rows out of {len(train_df)} ({(~outlier_mask).mean():.2%})")

train_df = train_df[outlier_mask].reset_index(drop=True)

Now, let's check the box plots once again

In [ ]:
make_box_plots(train_df[num_features])

Data looks more centered now.

### Correlation Matrix

Let's explore the correlation matrix among numerical features. This might tell us whether they are linearly independent or not.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = train_df[num_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

The correlation matrix shows no strong correlation among predictive features. This means we will keep them all.

### Predictive Power

In [ ]:
import math
import matplotlib.pyplot as plt
from predictPy import Analisis_Predictivo

cat_cols = [c for c in cat_features if c != "health_condition"]  # Excluir el target

ap = Analisis_Predictivo(
    datos=train_df,
    predecir="health_condition"
)

ncols = 2
nrows = math.ceil(len(cat_cols) / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 5*nrows),
    dpi=80
)

axes = axes.flatten()

for ax, feature in zip(axes, cat_cols):
    ap.poder_predictivo_categorica(
        var=feature,
        ax=ax
    )

# Eliminar ejes sobrantes
for ax in axes[len(cat_cols):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt

ap = Analisis_Predictivo(
    datos=train_df,
    predecir="health_condition"
)

# Excluir columnas que no quieras graficar
num_cols = [c for c in num_features if c != "id"]

ncols = 2
nrows = math.ceil(len(num_cols) / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 5 * nrows)
)

axes = axes.flatten()

for ax, feature in zip(axes, num_cols):
    ap.poder_predictivo_numerica(
        var=feature,
        ax=ax
    )

# Eliminar ejes sobrantes
for ax in axes[len(num_cols):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

### Information Gain / Mutual Information

Let's measure how much information each feature (numerical and categorical) provides about the target `health_condition`.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_features = [c for c in train_df.columns if c not in ['id', 'health_condition']]

X_mi = train_df[mi_features].copy()
discrete_mask = [col in cat_features for col in mi_features]

for col in mi_features:
    if col in cat_features:
        X_mi[col] = X_mi[col].cat.codes

mi_scores = mutual_info_classif(
    X_mi,
    train_df['health_condition'],
    discrete_features=discrete_mask,
    random_state=42
)

mi_df = (
    pd.Series(mi_scores, index=mi_features, name="Mutual Information")
    .sort_values(ascending=False)
    .to_frame()
)

plt.figure(figsize=(8, 6))
sns.barplot(x=mi_df["Mutual Information"], y=mi_df.index, orient="h")
plt.title("Mutual Information with health_condition")
plt.tight_layout()
plt.show()

In [ ]:
mi_df

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=train_df,
    x='health_condition',
    y='sleep_duration'
)

plt.title("sleep_duration vs health_condition")
plt.tight_layout()
plt.show()

In [ ]:
train_df.groupby("health_condition")["sleep_duration"].describe()

### UMAP

UMAP can graphically show us how the different clusters take shape, so that we can tell how well separated features are.

### Encoding categorical variables for UMAP

The previous UMAP only used `num_features` (numerical variables), leaving out `diet_type`, `stress_level`, `sleep_quality`, `physical_activity_level`, `smoking_alcohol` and `gender` — precisely the ones the Mutual Information / predictive power analysis showed to be relevant. We create a copy of `train_df` with these variables properly encoded:

* **Ordinal** (`stress_level`, `sleep_quality`, `physical_activity_level`, `smoking_alcohol`): these have a natural order, so they're encoded as integers 0/1/2 in that order instead of one-hot, to avoid losing the "less/more" notion.
* **Nominal** (`diet_type`, `gender`): no natural order, so they're one-hot encoded.
* **`Missing`**: has no valid ordinal position among the real categories, so for ordinal variables it's replaced with the mode and a `_was_missing` indicator is added (same pattern already used with `sleep_duration`), instead of assigning it an arbitrary rank.
* **Scaling**: continuous variables are standardized (`StandardScaler`) so they don't dominate the Euclidean distance over the binary/ordinal-encoded columns.

`id` and `health_condition` (the target) are excluded from the input set.

In [ ]:
from sklearn.preprocessing import StandardScaler

ordinal_specs = {
    'stress_level': ['low', 'medium', 'high'],
    'sleep_quality': ['poor', 'average', 'good'],
    'physical_activity_level': ['sedentary', 'moderate', 'active'],
    'smoking_alcohol': ['no', 'occasional', 'yes'],
}
ordinal_features = list(ordinal_specs)
nominal_features = ['diet_type', 'gender']

umap_df = train_df.copy()

for col in ordinal_features:
    umap_df[f'{col}_was_missing'] = (umap_df[col] == 'Missing').astype(int)
    mode_value = umap_df.loc[umap_df[col] != 'Missing', col].mode()[0]
    umap_df[col] = (
        umap_df[col]
            .replace('Missing', mode_value)
            .map({cat: i for i, cat in enumerate(ordinal_specs[col])})
    )

onehot = pd.get_dummies(umap_df[nominal_features], prefix=nominal_features, dtype=int)
umap_df = pd.concat(
    [umap_df.drop(columns=nominal_features + ['health_condition', 'id']), onehot],
    axis=1
)

binary_features = (
    ['sleep_duration_was_missing']
    + [f'{c}_was_missing' for c in ordinal_features]
    + list(onehot.columns)
)
continuous_features = umap_df.columns.difference(binary_features)

umap_df[continuous_features] = StandardScaler().fit_transform(umap_df[continuous_features])

umap_df.head()

In [ ]:
import os
import joblib
from umap import UMAP

umap_model_path = get_embedding_path('umap_model_encoded.joblib')

if os.path.exists(umap_model_path):
    print(f"Loading cached UMAP model from {umap_model_path}")
    umap_reducer = joblib.load(umap_model_path)
    embedding = umap_reducer.embedding_
else:
    print("No cached model found, fitting UMAP...")
    umap_reducer = UMAP(
        n_neighbors=30,
        min_dist=0.2,
    )
    embedding = umap_reducer.fit_transform(umap_df)

    os.makedirs(os.path.dirname(umap_model_path) or '.', exist_ok=True)
    joblib.dump(umap_reducer, umap_model_path)
    print(f"Saved UMAP model to {umap_model_path}")

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=embedding[:,0],
    y=embedding[:,1],
    hue=train_df['health_condition'],
    alpha=0.6
)

plt.show()

The UMAP trained on `umap_df` (numerical + encoded categorical features: `stress_level`, `sleep_quality`, `physical_activity_level`, `smoking_alcohol`, `diet_type`, `gender`) still shows no visually separated clusters by `health_condition`, and `at-risk` still dominates the plot (86% of rows).

However, there is a real quantitative improvement over the numeric-only UMAP:

| Metric | Numerical only | + encoded categoricals | Chance (imbalance baseline) |
|---|---|---|---|
| Silhouette score (`health_condition`) | -0.051 | -0.055 | — |
| kNN neighbor purity (k=15) | 75.5% | **85.4%** | 75.05% |

The silhouette score stays close to 0 in both cases: there are no global, well-delimited clusters separated by class. But local neighborhood purity increased notably once the encoded categoricals were included — well above the 75.05% that would be expected by pure chance given the class imbalance. This confirms that `stress_level` and the other categorical variables do carry real, local signal (consistent with their high Mutual Information), even though it isn't enough to produce sharp, visually separable boundaries in 2D. This is consistent with the fact that a 2D UMAP projection is lossy, and a classifier with access to the full feature space (not just 2D) will likely be able to exploit that signal better.